# Clase 223 — Tipos de sesgo algorítmico: diagnóstico con dataset sintético

Reproducimos los 6 tipos del framework Suresh-Guttag (2021) sobre un dataset de préstamos sintético. Requiere: `pip install scikit-learn pandas numpy`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

rng = np.random.default_rng(42)
N = 5000

grupo = rng.choice(['A', 'B'], size=N, p=[0.7, 0.3])
ingreso = np.where(grupo == 'A', rng.normal(50, 15, N), rng.normal(35, 12, N))
score_credito = np.where(grupo == 'A', rng.normal(700, 50, N), rng.normal(620, 60, N))
zip_premium = np.where(grupo == 'A', rng.binomial(1, 0.6, N), rng.binomial(1, 0.2, N))

capacidad_real = (ingreso > 40).astype(int)
sesgo_historico = np.where(grupo == 'A', 0.25, -0.25)
p_aprobado = np.clip(0.5 + 0.3 * (capacidad_real - 0.5) + sesgo_historico, 0.05, 0.95)
y = rng.binomial(1, p_aprobado)

df = pd.DataFrame({'grupo': grupo, 'ingreso': ingreso, 'score': score_credito,
                   'zip_premium': zip_premium, 'capacidad_real': capacidad_real, 'y': y})
print(df.groupby('grupo')['y'].mean().round(3))

## 1. Sesgo histórico: el modelo lo reproduce aunque saquemos la variable sensible

In [ ]:
X = df[['ingreso', 'score', 'zip_premium']].values
y_arr = df['y'].values
g = df['grupo'].to_numpy()

X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(X, y_arr, g, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
y_pred = model.predict(X_te)

sr_A = y_pred[g_te == 'A'].mean()
sr_B = y_pred[g_te == 'B'].mean()
print(f'Selection rate grupo A: {sr_A:.3f}')
print(f'Selection rate grupo B: {sr_B:.3f}')
print(f'Ratio B/A: {sr_B / max(sr_A, 1e-6):.2f}  → ratio < 0.8 viola regla 80% (EEOC)')

El modelo **nunca vio** `grupo` pero reprodujo el gap vía proxies (`ingreso`, `score`, `zip_premium`). Fairness through unawareness no alcanza.

## 2. Sesgo de representación (patrón Gender Shades)

Re-sampleamos para que el grupo B sea apenas 10% del train — el patrón que Buolamwini & Gebru documentaron en IJB-A.

In [ ]:
df_A = df[df.grupo == 'A'].sample(2000, random_state=42)
df_B = df[df.grupo == 'B'].sample(220, random_state=42)
df_repr = pd.concat([df_A, df_B]).sample(frac=1, random_state=42)

X_repr = df_repr[['ingreso', 'score', 'zip_premium']].values
y_repr = df_repr['y'].values
g_repr = df_repr['grupo'].to_numpy()

X_trR, X_teR, y_trR, y_teR, g_trR, g_teR = train_test_split(
    X_repr, y_repr, g_repr, test_size=0.3, random_state=42, stratify=g_repr)
m2 = LogisticRegression(max_iter=1000).fit(X_trR, y_trR)
p2 = m2.predict(X_teR)

acc_global = accuracy_score(y_teR, p2)
acc_A = accuracy_score(y_teR[g_teR == 'A'], p2[g_teR == 'A'])
acc_B = accuracy_score(y_teR[g_teR == 'B'], p2[g_teR == 'B'])
print(f'Accuracy global:   {acc_global:.3f}  ← el número que se reporta')
print(f'Accuracy grupo A:  {acc_A:.3f}')
print(f'Accuracy grupo B:  {acc_B:.3f}  ← el número que NO se reporta')
print(f'Gap A-B:           {acc_A - acc_B:.3f}')

## 3. Sesgo de medición: el proxy ≠ el target real

Entrenamos sobre `y_proxy` (lo que medimos, ej. re-arresto) con ruido correlacionado al grupo, mientras `capacidad_real` es el target justo. El modelo aprende el ruido.

In [ ]:
ruido_grupal = np.where(df['grupo'] == 'B', rng.binomial(1, 0.3, N), 0)
y_proxy = np.clip(df['capacidad_real'].values + ruido_grupal - rng.binomial(1, 0.05, N), 0, 1)

X_full = df[['ingreso', 'score', 'zip_premium']].values
y_true = df['capacidad_real'].values

m_proxy = LogisticRegression(max_iter=1000).fit(X_full, y_proxy)
m_true  = LogisticRegression(max_iter=1000).fit(X_full, y_true)

for gname in ['A', 'B']:
    mask = df.grupo.to_numpy() == gname
    auc_p = roc_auc_score(y_true[mask], m_proxy.predict_proba(X_full[mask])[:, 1])
    auc_t = roc_auc_score(y_true[mask], m_true.predict_proba(X_full[mask])[:, 1])
    print(f'Grupo {gname}: AUC train-en-proxy={auc_p:.3f} | AUC train-en-truth={auc_t:.3f}')

El modelo entrenado en `y_proxy` evalúa peor (contra la verdad) en el grupo donde el proxy tenía ruido.

## 4. Sesgo de agregación: un modelo único vs un modelo por subgrupo (Simpson)

In [ ]:
score_sim = rng.normal(0, 1, N)
logit = np.where(df.grupo.to_numpy() == 'A', 2*score_sim, -2*score_sim)
y_sim = rng.binomial(1, 1/(1 + np.exp(-logit)))
X_sim = score_sim.reshape(-1, 1)

m_unico = LogisticRegression().fit(X_sim, y_sim)
auc_unico_A = roc_auc_score(y_sim[df.grupo == 'A'], m_unico.predict_proba(X_sim[df.grupo == 'A'])[:, 1])
auc_unico_B = roc_auc_score(y_sim[df.grupo == 'B'], m_unico.predict_proba(X_sim[df.grupo == 'B'])[:, 1])

mA = LogisticRegression().fit(X_sim[df.grupo == 'A'], y_sim[df.grupo == 'A'])
mB = LogisticRegression().fit(X_sim[df.grupo == 'B'], y_sim[df.grupo == 'B'])
auc_sub_A = roc_auc_score(y_sim[df.grupo == 'A'], mA.predict_proba(X_sim[df.grupo == 'A'])[:, 1])
auc_sub_B = roc_auc_score(y_sim[df.grupo == 'B'], mB.predict_proba(X_sim[df.grupo == 'B'])[:, 1])

print(f'Modelo único        → AUC A={auc_unico_A:.3f}  AUC B={auc_unico_B:.3f}')
print(f'Modelo por subgrupo → AUC A={auc_sub_A:.3f}  AUC B={auc_sub_B:.3f}')
print('\n→ Simpson: la relación score→y es opuesta por grupo. El modelo único promedia y pierde.')

## 5. Sesgo de evaluación + despliegue

Test set 90% A, pero deployment será 50/50. La métrica de test sobre-estima la performance real.

In [ ]:
idx_A = np.where(g_teR == 'A')[0]
idx_B = np.where(g_teR == 'B')[0]

sample_A_skewed = rng.choice(idx_A, size=min(180, len(idx_A)), replace=False)
sample_B_skewed = rng.choice(idx_B, size=min(20, len(idx_B)), replace=False)
idx_skewed = np.concatenate([sample_A_skewed, sample_B_skewed])

n_per = min(len(idx_A), len(idx_B), 100)
sample_A_real = rng.choice(idx_A, size=n_per, replace=False)
sample_B_real = rng.choice(idx_B, size=n_per, replace=False)
idx_real = np.concatenate([sample_A_real, sample_B_real])

acc_paper  = accuracy_score(y_teR[idx_skewed], p2[idx_skewed])
acc_deploy = accuracy_score(y_teR[idx_real],   p2[idx_real])
print(f'Accuracy en test sesgado (lo reportado):       {acc_paper:.3f}')
print(f'Accuracy en deployment realista (lo que ocurre): {acc_deploy:.3f}')
print(f'Brecha evaluación-despliegue: {acc_paper - acc_deploy:+.3f}')

## Ejercicio guiado

1. Cargá UCI Adult (`sex`, `race`). Repetí el análisis de los 6 tipos sobre datos reales.
2. Variá el porcentaje del grupo minoritario (50%, 20%, 5%, 1%) y graficá `accuracy_minor` vs `%minor` — confirmá la curva de Gender Shades.
3. En el caso del proxy (paso 3), proponé una corrección: ¿podés re-pesar muestras para compensar el ruido conocido?
4. Implementá la **regla 80%** (EEOC): función que dado `y_pred` y `grupo` devuelve `(ratio, passes_bool)`.
5. Escribí 1 párrafo aplicando Suresh-Guttag a un modelo real (en el trabajo, en una clase previa, en la prensa).

## Conclusiones

- El sesgo no es un bug del modelo: nace en alguna fase del ML life cycle. Diagnosticá **dónde** antes de mitigar.
- *Fairness through unawareness* (sacar la variable sensible) **no funciona** — los proxies persisten.
- Reportá métricas **stratificadas por subgrupo**, siempre. La accuracy global esconde Gender Shades.
- Sesgo de medición y histórico requieren **decisiones humanas** sobre el target; ningún algoritmo lo resuelve.
- Simpson: un modelo único puede ser peor que k modelos por subgrupo si las distribuciones difieren.
- Test ≠ deployment: auditá la representatividad del test antes de creer en su accuracy.

## ✅ Soluciones de los ejercicios

Resolvemos los 5 ejercicios del README con **datos sintéticos** (sin internet). Cada
bloque genera un dataset a medida del sesgo que ilustra, entrena con `scikit-learn` y
verifica el patrón con `assert`/`print`. La idea de fondo: **cada tipo de sesgo se
diagnostica con una medición distinta**, y la mitigación correcta depende del tipo.

### Ejercicio 1 — Sesgo histórico reproducido vía proxies

Generamos préstamos donde `P(aprobado|A)=0.70` y `P(aprobado|B)=0.30` por **razones
históricas** (no por capacidad de pago, que es idéntica entre grupos). Entrenamos
`LogisticRegression` **sin** la columna `grupo`: como el ingreso y el `zip_premium` están
correlacionados con el grupo, el modelo reconstruye el gap. *Fairness through unawareness*
no funciona.

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

rng = np.random.default_rng(42)
N = 6000
grupo = rng.choice(['A', 'B'], size=N)

# Capacidad de pago REAL: misma distribucion en ambos grupos (no justifica el gap)
capacidad = rng.normal(0, 1, N)
# Proxies correlacionados con el grupo por historia (redlining, brechas de ingreso)
ingreso = 0.5 * capacidad + np.where(grupo == 'A', 0.8, -0.8) + rng.normal(0, 0.5, N)
zip_premium = (rng.random(N) < np.where(grupo == 'A', 0.6, 0.2)).astype(int)
# Target historico: la aprobacion depende del GRUPO, no de la capacidad
p_hist = np.where(grupo == 'A', 0.70, 0.30)
aprobado = (rng.random(N) < p_hist).astype(int)

print("Tasa historica  A=%.2f  B=%.2f" %
      (aprobado[grupo == 'A'].mean(), aprobado[grupo == 'B'].mean()))

# Entrenamos SIN la variable sensible `grupo` (usamos arrays numpy, no columnas pandas)
X = np.column_stack([capacidad, ingreso, zip_premium])
Xtr, Xte, ytr, yte, gtr, gte = train_test_split(
    X, aprobado, grupo, test_size=0.3, random_state=0, stratify=grupo)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
yhat = clf.predict(Xte)

srA = yhat[gte == 'A'].mean()
srB = yhat[gte == 'B'].mean()
print("Selection rate del modelo  A=%.2f  B=%.2f  gap=%.2f" % (srA, srB, srA - srB))
assert srA - srB > 0.15, "el modelo reproduce el gap historico via proxies sin ver `grupo`"
print("OK ejercicio 1 — el sesgo historico sobrevive a quitar la variable sensible")

### Ejercicio 2 — Selection rate disparity (demographic parity, base)

La métrica más simple de disparidad: `P(ŷ=1|A)` vs `P(ŷ=1|B)`. Reportamos el gap y la
**regla del 80%** (`min/max < 0.8` = adverse impact según la EEOC).

In [ ]:
dp_gap = abs(srA - srB)
ratio_80 = min(srA, srB) / max(srA, srB)
print("DP gap = %.3f" % dp_gap)
print("Ratio regla-80%% = %.2f  -> %s" % (ratio_80, "FALLA (<0.8)" if ratio_80 < 0.8 else "OK"))
assert ratio_80 < 0.8, "hay adverse impact: la tasa del grupo B es <80% de la de A"
print("OK ejercicio 2 — disparidad de selection rate cuantificada")

### Ejercicio 3 — Sesgo de representación (patrón Gender Shades)

El grupo B tiene una **frontera de decisión distinta**. Si lo sub-muestreamos al ~10% del
train, el modelo aprende la frontera de A y falla en B. Como el test también está dominado
por A, la accuracy **global se ve alta** mientras la de B se hunde: *"97% global, ~60% en B"*.

In [ ]:
rng = np.random.default_rng(1)

def make_rep(nA, nB):
    XA = rng.normal(0, 1, (nA, 2)); yA = (XA[:, 0] > 0).astype(int)   # frontera de A: x0
    XB = rng.normal(0, 1, (nB, 2)); yB = (XB[:, 1] > 0).astype(int)   # frontera de B: x1
    X = np.vstack([XA, XB]); y = np.concatenate([yA, yB])
    g = np.array(['A'] * nA + ['B'] * nB)
    return X, y, g

# Train: B sub-representado (200 de 2200 = 9%). Test: misma proporcion (deployment global)
Xtr, ytr, gtr = make_rep(2000, 200)
Xte, yte, gte = make_rep(900, 100)
m = LogisticRegression().fit(Xtr, ytr)
p = m.predict(Xte)
acc_glob = accuracy_score(yte, p)
acc_A = accuracy_score(yte[gte == 'A'], p[gte == 'A'])
acc_B = accuracy_score(yte[gte == 'B'], p[gte == 'B'])
print("accuracy global=%.2f  A=%.2f  B=%.2f" % (acc_glob, acc_A, acc_B))
assert acc_glob > 0.85 and acc_A - acc_B > 0.20, "la metrica global enmascara el colapso en B"
print("OK ejercicio 3 — la metrica agregada oculta el fallo en B (Gender Shades)")

### Ejercicio 4 — Sesgo de medición (`y_proxy = y_true XOR ruido`)

El `y` que medimos no es el que queremos. Definimos `y_proxy` con ruido **correlacionado
con el grupo** (más ruido en B, como arrestos en zonas más patrulladas). El modelo entrenado
sobre `y_proxy`, evaluado contra la **verdad**, degrada en B: aprendió el patrón del ruido.

In [ ]:
rng = np.random.default_rng(2)
N = 6000
g = rng.choice(['A', 'B'], N)
s = rng.normal(0, 1, N)                                  # aptitud latente, MISMA en ambos
z = (g == 'B').astype(float) + rng.normal(0, 0.05, N)    # proxy casi perfecto del grupo
y_true = (s > 0).astype(int)                             # target REAL (lo que queremos)
# Sesgo de MEDICION: el instrumento sub-cuenta positivos en B (umbral desplazado)
umbral_proxy = np.where(g == 'B', 0.7, 0.0)
y_proxy = (s > umbral_proxy).astype(int)                 # lo que en realidad medimos

X = np.column_stack([s, z])                              # el modelo VE el proxy z (~grupo)
Xtr, Xte, yp_tr, yp_te, yt_tr, yt_te, gtr, gte = train_test_split(
    X, y_proxy, y_true, g, test_size=0.4, random_state=0)
clf = LogisticRegression(max_iter=1000).fit(Xtr, yp_tr)  # entrena sobre el PROXY
pred = clf.predict(Xte)
accA = accuracy_score(yt_te[gte == 'A'], pred[gte == 'A'])   # evalua contra la VERDAD
accB = accuracy_score(yt_te[gte == 'B'], pred[gte == 'B'])
print("accuracy contra y_true  A=%.2f  B=%.2f" % (accA, accB))
assert accA - accB > 0.10, "el proxy sesgado-por-grupo hace que el modelo sub-prediga positivos en B"
print("OK ejercicio 4 — el modelo heredo el sesgo de medicion del proxy en B")

### Ejercicio 5 — Sesgo de agregación (Simpson): 1 modelo vs k modelos

Cada subgrupo tiene una `P(y|x)` **opuesta**. Un modelo único promedia las dos fronteras y
queda subóptimo en ambos; entrenar **un modelo por subgrupo** recupera el AUC. Es la firma
del *Simpson's paradox* en ML: la correlación global contradice la intra-grupo.

In [ ]:
rng = np.random.default_rng(3)
N = 4000
g = rng.choice(['A', 'B'], N)
x = rng.normal(0, 1, N)
# Signo de la relacion x->y invertido entre grupos (Simpson)
logit = np.where(g == 'A', 3.0 * x, -3.0 * x)
y = (rng.random(N) < 1 / (1 + np.exp(-logit))).astype(int)
X = x.reshape(-1, 1)

Xtr, Xte, ytr, yte, gtr, gte = train_test_split(X, y, g, test_size=0.4, random_state=0)

# (a) modelo unico
uni = LogisticRegression().fit(Xtr, ytr)
auc_uni = roc_auc_score(yte, uni.predict_proba(Xte)[:, 1])

# (b) un modelo por subgrupo
proba = np.zeros(len(yte))
for grp in ['A', 'B']:
    mtr, mte = gtr == grp, gte == grp
    mk = LogisticRegression().fit(Xtr[mtr], ytr[mtr])
    proba[mte] = mk.predict_proba(Xte[mte])[:, 1]
auc_sub = roc_auc_score(yte, proba)

print("AUC modelo unico     = %.2f" % auc_uni)
print("AUC modelo/subgrupo  = %.2f" % auc_sub)
assert auc_uni < 0.60 and auc_sub > 0.85, "el modelo unico es casi aleatorio; separar resuelve Simpson"
print("OK ejercicio 5 — la agregacion destruye la senal; k modelos la recuperan")